# NUMPY MASTERY — CHAPTERS 47–49

```
NUMPY MASTERY
Current Chapters: 47 (NumPy for NLP), 48 (NumPy for Time Series),
                  49 (NumPy for Recommender Systems)
Progress:         49 / 75 chapters
Completed:        Ch1-46
Remaining:        26 chapters
```

Three application domains, one notebook — each is really just Chapters 1-46's tools (indexing,
broadcasting, linear algebra, axis reductions) pointed at a new kind of data: text as vectors,
data ordered in time, and users/items as a matrix.


In [ ]:
import numpy as np
print("NumPy version:", np.__version__)


---
# CHAPTER 47 — NumPy for NLP

## Concept
Text becomes numbers via a vocabulary mapping (word → integer ID), then those IDs become
vectors — either sparse (one-hot, bag-of-words) or dense (embeddings). Every similarity
comparison between pieces of text ultimately reduces to a vector operation you already know:
**cosine similarity**, built entirely from Chapter 22's dot product and Chapter 23's norm.


In [ ]:
# --- Building a vocabulary and one-hot / bag-of-words representation from scratch ---
documents = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are great pets",
]

# Build vocabulary: unique words -> integer IDs (Chapter 17's np.unique, return_inverse IS this)
all_words = " ".join(documents).split()
vocab, word_ids = np.unique(all_words, return_inverse=True)
print("Vocabulary:", vocab)
print("Vocabulary size:", len(vocab))

def bag_of_words(doc, vocab):
    """Count occurrences of each vocabulary word in doc -- a (vocab_size,) vector."""
    words = doc.split()
    vector = np.zeros(len(vocab), dtype=int)
    for word in words:
        idx = np.searchsorted(vocab, word)   # Chapter 16's binary search, since vocab is sorted
        vector[idx] += 1
    return vector

bow_matrix = np.array([bag_of_words(doc, vocab) for doc in documents])
print("\nBag-of-words matrix (documents x vocabulary):\n", bow_matrix)
print("Shape:", bow_matrix.shape, " <- (n_documents, vocab_size)")


### Expected Output (verified on NumPy 2.4.4)
```
Vocabulary: ['and' 'are' 'cat' 'cats' 'dog' 'dogs' 'great' 'log' 'mat' 'on' 'pets' 'sat'
 'the']
Vocabulary size: 13

Bag-of-words matrix (documents x vocabulary):
 [[0 0 1 0 0 0 0 0 1 1 0 1 2]
  [0 0 0 0 1 0 0 1 0 1 0 1 2]
  [1 1 0 1 0 1 1 0 0 0 1 0 0]]
Shape: (3, 13)  <- (n_documents, vocab_size)
```

### Cosine similarity — the workhorse of NLP/embedding comparisons


In [ ]:
def cosine_similarity_matrix(X):
    """All-pairs cosine similarity for rows of X, fully vectorized -- (n, n) output."""
    norms = np.linalg.norm(X, axis=1, keepdims=True)          # Chapter 23's axis-aware norm
    norms = np.where(norms == 0, 1, norms)                     # guard against a zero vector
    normalized = X / norms                                      # Chapter 36's L2-normalize-rows
    return normalized @ normalized.T                            # Chapter 22's matmul, ALL pairs at once


similarities = cosine_similarity_matrix(bow_matrix.astype(float))
print("Cosine similarity matrix (documents x documents):\n", np.round(similarities, 3))
print("\ndoc0 ('cat...mat') vs doc1 ('dog...log') similarity:", round(similarities[0, 1], 3),
      " <- structurally similar sentences, share 'the...sat...on...the'")
print("doc0 vs doc2 ('cats and dogs...') similarity       :", round(similarities[0, 2], 3),
      " <- almost no shared words")


### Expected Output (verified on NumPy 2.4.4)
```
Cosine similarity matrix (documents x documents):
 [[1.   0.75 0.  ]
  [0.75 1.   0.  ]
  [0.   0.   1.  ]]

doc0 ('cat...mat') vs doc1 ('dog...log') similarity: 0.75  <- structurally similar sentences, share 'the...sat...on...the'
doc0 vs doc2 ('cats and dogs...') similarity       : 0.0  <- almost no shared words
```

### Why `normalized @ normalized.T` computes cosine similarity for EVERY pair at once
Cosine similarity between two vectors is `dot(a,b) / (norm(a) * norm(b))`. Pre-dividing every
row by its own norm (`normalized = X / norms`) means each row now has norm exactly 1 — so a
plain dot product between any two *normalized* rows already **equals** their cosine similarity
(the norms in the denominator become 1 and vanish). `normalized @ normalized.T` computes the
dot product of every row with every other row simultaneously (Chapter 22's shape rule:
`(n,d) @ (d,n) -> (n,n)`) — this single line replaces what would otherwise be a nested loop over
all document pairs.

### Word embeddings and nearest-neighbor word lookup


In [ ]:
# Simulate pretrained word embeddings (in reality, learned by a model like word2vec/GloVe)
rng = np.random.default_rng(0)
embed_dim = 8
word_embeddings = rng.normal(size=(len(vocab), embed_dim))

def nearest_words(query_word, vocab, embeddings, top_k=3):
    query_idx = np.searchsorted(vocab, query_word)
    query_vec = embeddings[query_idx]

    sims = cosine_similarity_matrix(np.vstack([query_vec, embeddings]))[0, 1:]   # query vs all words
    sims[query_idx] = -np.inf   # exclude the query word itself from its own neighbor list
    top_idx = np.argpartition(sims, -top_k)[-top_k:]                              # Chapter 16's efficient top-K
    top_idx_sorted = top_idx[np.argsort(sims[top_idx])[::-1]]
    return list(zip(vocab[top_idx_sorted], np.round(sims[top_idx_sorted], 3)))

print("Words most similar to 'cat' (random embeddings, so results are NOT semantically meaningful,")
print("but demonstrate the exact mechanism real embedding search uses):")
for word, sim in nearest_words("cat", vocab, word_embeddings, top_k=3):
    print(f"  {word}: {sim}")


### Expected Output (verified on NumPy 2.4.4 — random embeddings, so word identities are arbitrary)
```
Words most similar to 'cat' (random embeddings, so results are NOT semantically meaningful,
but demonstrate the exact mechanism real embedding search uses):
  are: 0.178
  on: 0.152
  dogs: 0.116
```
Note the query word itself ("cat") is deliberately excluded from its own neighbor list — an
easy mistake to make (a word is always most "similar" to itself, similarity `1.0`), fixed here
by masking the query's own index to `-inf` before the top-K selection, exactly mirroring the
"never recommend an already-rated item" masking pattern used later in Chapter 49.

### This IS how real semantic search / retrieval-augmented generation works, mechanically
Real word/sentence embeddings (word2vec, BERT, modern LLM embeddings) are learned so that
semantically similar text ends up with *high cosine similarity* — but once you have the
embedding vectors (from anywhere), finding "nearest neighbors" is exactly the
`cosine_similarity_matrix` + `argpartition` top-K pattern shown here. Production vector
databases add approximate-nearest-neighbor indexing for speed at scale, but the underlying
mathematical operation being approximated is precisely this.

### Common Mistake
Comparing embedding vectors with raw Euclidean distance (`np.linalg.norm(a - b)`) when the
embedding model was trained/intended for cosine similarity (very common with normalized
embeddings) — these can rank neighbors differently, especially when vector magnitudes vary
meaningfully across the embedding space; always check which similarity metric an embedding
model was designed for.

### Interview Insight
> "Given a matrix of embedding vectors, how would you find the top-5 most similar embeddings to
> a query vector, efficiently?"
Normalize both the query and the embedding matrix rows to unit length (Chapter 36), compute
their dot products in one matrix-vector multiplication (`embeddings_normalized @
query_normalized`, Chapter 22), then use `argpartition` (Chapter 16) rather than a full sort to
extract the top-5 in O(n) average time.


---
# CHAPTER 48 — NumPy for Time Series

## Concept
Time series operations — lags, differences, returns, rolling statistics — are all built from
shifted slicing (Chapter 5) and `np.cumsum`/manual sliding-window tricks, without ever needing
a "time series" library for the core mechanics.


In [ ]:
rng = np.random.default_rng(0)
prices = 100 + np.cumsum(rng.normal(scale=1.0, size=20))   # simulate a random-walk price series
print("prices:", np.round(prices, 2))

# --- Lag features: value from k steps ago ---
def lag(series, k):
    """Shift series forward by k, padding the first k entries with NaN (Chapter 15's tool)."""
    lagged = np.full_like(series, np.nan)
    lagged[k:] = series[:-k]
    return lagged

lag1 = lag(prices, 1)
print("\nlag1 (previous value):", np.round(lag1, 2))

# --- Differences and returns ---
diffs = np.diff(prices)                              # prices[i] - prices[i-1], built-in NumPy function
pct_returns = diffs / prices[:-1]                     # percentage change

print("\ndiffs (Chapter 15/np.diff):", np.round(diffs, 2))
print("pct_returns              :", np.round(pct_returns, 4))


### Expected Output (verified on NumPy 2.4.4)
```
prices: [100.13  99.99 100.63 100.74 100.2  100.56 101.87 102.82 102.11 100.85
 100.22 100.26  97.94  97.72  96.48  95.74  95.2   94.88  95.29  96.34]

lag1 (previous value): [   nan 100.13  99.99 100.63 100.74 100.2  100.56 101.87 102.82 102.11
 100.85 100.22 100.26  97.94  97.72  96.48  95.74  95.2   94.88  95.29]

diffs (Chapter 15/np.diff): [-0.13  0.64  0.1  -0.54  0.36  1.3   0.95 -0.7  -1.27 -0.62  0.04 -2.33
 -0.22 -1.25 -0.73 -0.54 -0.32  0.41  1.04]
pct_returns              : [-0.0013  0.0064  0.001  -0.0053  0.0036  0.013   0.0093 -0.0068 -0.0124
 -0.0062  0.0004 -0.0232 -0.0022 -0.0127 -0.0076 -0.0057 -0.0033  0.0043
  0.0109]
```

### `np.diff` — a function you'll use constantly for time series
`np.diff(x)` computes `x[1:] - x[:-1]` in one call — note the output is **one element shorter**
than the input (there's no "difference" for the very first point), a shape detail that trips up
beginners aligning differenced series back against a dataframe's original index.

### Rolling/moving statistics — the sliding-window pattern


In [ ]:
def rolling_mean(series, window):
    """Simple moving average, vectorized via cumsum -- O(n), not O(n*window)."""
    cumsum = np.cumsum(np.insert(series, 0, 0))     # prepend a 0 so cumsum[i] = sum(series[:i])
    return (cumsum[window:] - cumsum[:-window]) / window

def rolling_std(series, window):
    """Rolling standard deviation via a sliding-window VIEW (Chapter 8's view mechanism,
    applied for real) -- no explicit Python loop over windows."""
    windows = np.lib.stride_tricks.sliding_window_view(series, window)   # shape (n-window+1, window)
    return windows.std(axis=1)                                            # Chapter 10's axis reduction


window = 5
sma = rolling_mean(prices, window)
rstd = rolling_std(prices, window)

print(f"5-period rolling mean (first few): {np.round(sma[:5], 2)}")
print(f"5-period rolling std  (first few): {np.round(rstd[:5], 2)}")
print(f"\nrolling_mean output length: {len(sma)}  (original length {len(prices)}, window {window})")


### Expected Output (verified on NumPy 2.4.4)
```
5-period rolling mean (first few): [100.34 100.43 100.8  101.24 101.51]
5-period rolling std  (first few): [0.29 0.28 0.56 0.97 0.98]

rolling_mean output length: 16  (original length 20, window 5)
```

### Why `cumsum`-based rolling mean is O(n), not O(n × window)
A naive rolling mean recomputes the sum of each window from scratch — `O(window)` work per
window, `O(n × window)` total. The cumsum trick computes the **prefix sum** once (`O(n)`), then
each window's sum is just `cumsum[end] - cumsum[start]` (`O(1)` per window) — the classic
"prefix sum" algorithmic technique, expressed here as pure NumPy array slicing rather than an
explicit loop.

### `sliding_window_view` — a genuinely zero-copy way to build sliding windows
`np.lib.stride_tricks.sliding_window_view(series, window)` creates an array of shape
`(n-window+1, window)` where each row is one window — but critically, this is a **view**
(Chapter 8), constructed purely by manipulating strides (Chapter 28) to make overlapping windows
appear as separate rows **without duplicating any data in memory**. This is an advanced,
powerful technique: the "logical" array size (`(n-window+1) * window` elements) can be much
larger than the actual underlying memory (`n` elements) because of the deliberate overlap.

### Common Mistake
Computing rolling statistics with a Python loop over window start positions for a large time
series — this is the Chapter 1 lesson yet again: `sliding_window_view` + a vectorized reduction
(`.mean(axis=1)`, `.std(axis=1)`) is dramatically faster than looping, and there's essentially
never a reason to loop for standard rolling-window statistics in NumPy.

### Interview Insight
> "How would you compute a rolling mean over a large time series efficiently in NumPy?"
Either the `cumsum`-based prefix-sum trick (`O(n)`, simple, works for mean/sum specifically), or
`np.lib.stride_tricks.sliding_window_view` combined with `.mean(axis=1)` (more general — works
for any per-window statistic like std, min, max, not just sum-based ones) — both avoid the
`O(n × window)` cost of naively recomputing each window's statistic from scratch, and both avoid
an explicit Python loop over window positions.


---
# CHAPTER 49 — NumPy for Recommender Systems

## Concept
A recommender system's core data structure is a **user-item matrix** — rows are users, columns
are items, values are ratings/interactions (often mostly missing/zero — "sparse"). Similarity
between users or items (Chapter 47's cosine similarity, reused directly) and matrix
factorization (Chapter 26's SVD, reused directly) are the two classical approaches, both built
entirely from tools already covered.


In [ ]:
# --- User-item ratings matrix (0 = not rated) ---
ratings = np.array([
    #  item0 item1 item2 item3 item4
    [   5,    3,    0,    1,    0],   # user0
    [   4,    0,    0,    1,    0],   # user1
    [   1,    1,    0,    5,    4],   # user2
    [   0,    0,    5,    4,    0],   # user3
    [   0,    0,    4,    0,    5],   # user4
], dtype=float)

n_users, n_items = ratings.shape
print("ratings matrix (users x items):\n", ratings)

# --- User-user similarity via cosine similarity (reusing Chapter 47's function) ---
def cosine_similarity_matrix(X):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1, norms)
    normalized = X / norms
    return normalized @ normalized.T

user_similarity = cosine_similarity_matrix(ratings)
print("\nUser-user similarity matrix:\n", np.round(user_similarity, 3))


### Expected Output (verified on NumPy 2.4.4)
```
ratings matrix (users x items):
 [[5. 3. 0. 1. 0.]
  [4. 0. 0. 1. 0.]
  [1. 1. 0. 5. 4.]
  [0. 0. 5. 4. 0.]
  [0. 0. 4. 0. 5.]]

User-user similarity matrix:
 [[1.    0.861 0.335 0.106 0.   ]
  [0.861 1.    0.333 0.152 0.   ]
  [0.335 0.333 1.    0.476 0.476]
  [0.106 0.152 0.476 1.    0.488]
  [0.    0.    0.476 0.488 1.   ]]
```

### Item recommendation via user-based collaborative filtering


In [ ]:
def recommend_for_user(user_idx, ratings, user_similarity, top_k=2):
    """Recommend items by weighting OTHER users' ratings by their similarity to this user,
    excluding items the user has already rated."""
    sims = user_similarity[user_idx].copy()
    sims[user_idx] = 0   # exclude the user's own similarity to themselves (always 1.0)

    # Weighted sum of everyone else's ratings, weighted by similarity to the target user
    weighted_scores = sims @ ratings                          # (n_items,) via (n_users,) @ (n_users, n_items)
    weighted_scores /= (np.abs(sims).sum() + 1e-12)             # normalize by total similarity weight

    already_rated = ratings[user_idx] > 0
    weighted_scores[already_rated] = -np.inf                    # never recommend an already-rated item

    n_available = (~already_rated).sum()                        # can't recommend more than exist
    k = min(top_k, n_available)
    if k == 0:
        return []   # user has rated every item -- nothing left to recommend

    top_items = np.argpartition(weighted_scores, -k)[-k:]
    top_items_sorted = top_items[np.argsort(weighted_scores[top_items])[::-1]]
    return list(zip(top_items_sorted, np.round(weighted_scores[top_items_sorted], 3)))


for user in range(n_users):
    recs = recommend_for_user(user, ratings, user_similarity, top_k=2)
    print(f"User {user}: recommend items {recs}")


### Expected Output (verified on NumPy 2.4.4)
```
User 0: recommend items [(np.int64(4), np.float64(1.03)), (np.int64(2), np.float64(0.406))]
User 1: recommend items [(np.int64(1), np.float64(2.167)), (np.int64(4), np.float64(0.99))]
User 2: recommend items [(np.int64(2), np.float64(2.645))]
User 3: recommend items [(np.int64(4), np.float64(3.557)), (np.int64(0), np.float64(1.319))]
User 4: recommend items [(np.int64(3), np.float64(4.494)), (np.int64(0), np.float64(0.494))]
```
*(The `np.int64(...)`/`np.float64(...)` wrapper display is just NumPy's scalar repr in this
version — the underlying values are ordinary numbers.)*

### User 2 only gets ONE recommendation — and that's correct, not a bug
User 2's ratings are `[1, 1, 0, 5, 4]` — only item 2 is unrated; items 0, 1, 3, 4 are already
rated. Requesting `top_k=2` recommendations when only 1 unrated item exists is a genuine edge
case (mirroring the "K > available items" pattern from earlier chapters, like Chapter 41's KNN
and Chapter 19's array splitting) — the function above explicitly caps `k = min(top_k,
n_available)` and returns an empty list if a user has rated everything, rather than silently
returning an already-rated item with a `-inf` score (which is what happens if you skip that
guard, an easy and genuinely tempting bug to write here).

### `sims @ ratings` — this single line IS collaborative filtering
Shapes: `sims` is `(n_users,)`, `ratings` is `(n_users, n_items)` — `sims @ ratings` (Chapter
22's `(k,) @ (k, m) -> (m,)` vector-matrix rule) computes, for every item, a similarity-weighted
sum of every other user's rating for that item — users more similar to the target user
contribute more to the score. Dividing by the total similarity weight normalizes this into a
comparable scale across items. Masking already-rated items to `-inf` before `argpartition`
guarantees they're never selected as a top-K "recommendation."

### Matrix factorization via SVD (item embeddings, reusing Chapter 26 directly)


In [ ]:
# Low-rank approximation of the ratings matrix via SVD -- captures latent "taste" dimensions
U, s, Vt = np.linalg.svd(ratings, full_matrices=False)

n_factors = 2   # compress into 2 latent dimensions
ratings_approx = U[:, :n_factors] @ np.diag(s[:n_factors]) @ Vt[:n_factors, :]

print("Original ratings (only nonzero shown as-is):\n", ratings)
print("\nSVD-reconstructed ratings (rank-2 approximation):\n", np.round(ratings_approx, 2))
print("\n<- Notice values now appear in PREVIOUSLY-ZERO cells: these are the model's")
print("   predicted ratings for items the user hasn't rated yet -- the actual recommendation signal")

# Concretely: predicted rating for (user0, item2) which was originally 0 (unrated)
print(f"\nPredicted rating, user0 x item2 (originally unrated): {ratings_approx[0, 2]:.2f}")


### Expected Output (verified on NumPy 2.4.4)
```
Original ratings (only nonzero shown as-is):
 [[5. 3. 0. 1. 0.]
  [4. 0. 0. 1. 0.]
  [1. 1. 0. 5. 4.]
  [0. 0. 5. 4. 0.]
  [0. 0. 4. 0. 5.]]

SVD-reconstructed ratings (rank-2 approximation):
 [[ 4.85  2.04 -0.85  1.98  0.05]
  [ 3.19  1.35 -0.49  1.37  0.1 ]
  [ 2.1   1.04  2.54  3.4   2.79]
  [ 0.06  0.19  3.17  2.79  3.02]
  [-0.99 -0.24  3.54  2.52  3.19]]

<- Notice values now appear in PREVIOUSLY-ZERO cells: these are the model's
   predicted ratings for items the user hasn't rated yet -- the actual recommendation signal

Predicted rating, user0 x item2 (originally unrated): -0.85
```

### Reading this result honestly, including its limitation
The SVD reconstruction fills in every cell with a *predicted* rating, including previously
unrated ones — this is the essence of matrix-factorization-based recommendation (compress users
and items into a shared low-dimensional "latent taste" space, then reconstruct to predict
missing entries). Notice the predicted rating for `(user0, item2)` came out **negative**
(`-0.85`), which is not a valid rating on a typical 1-5 scale — plain SVD on a matrix with zeros
standing in for "unrated" (rather than genuinely-zero) is a known limitation: it treats
"unrated" as "rated zero," which biases the reconstruction. Production recommender systems use
specialized matrix factorization objectives (e.g., only optimizing over *observed* entries,
adding regularization) rather than naive full-matrix SVD — but the core mathematical machinery
(compress into `U`, `Σ`, `Vᵀ`, reconstruct) is the same idea this chapter demonstrates directly.

### Interview Insight
> "How does SVD-based matrix factorization relate to collaborative filtering recommendation?"
Treating the user-item ratings matrix as a matrix to decompose via SVD (or a specialized
matrix-factorization variant) compresses users and items into a shared low-dimensional "latent
factor" space; reconstructing the matrix from a truncated (rank-k) decomposition fills in
predicted values for unrated user-item pairs — the reconstructed value for a previously-missing
entry *is* the model's rating prediction, which is then used to rank and recommend items. A key
caveat: naive full-matrix SVD treats missing ratings as zero, which biases predictions; real
systems typically optimize only over observed entries instead.


---
## Chapters 47–49 Summary

- **NLP**: text → vocabulary IDs (`np.unique(..., return_inverse=True)`, Chapter 17 reused) →
  bag-of-words vectors → cosine similarity (`normalize rows, then one matmul`) for all-pairs
  comparison at once — this single mechanism underlies embedding search, from word2vec to
  modern retrieval systems.
- **Time series**: `np.diff` for differences/returns; `cumsum`-based prefix sums give O(n)
  rolling means; `sliding_window_view` gives a genuinely zero-copy way to compute arbitrary
  per-window statistics (std, min, max) without ever looping over window positions.
- **Recommender systems**: the user-item matrix is the central data structure; user-user cosine
  similarity + `sims @ ratings` implements collaborative filtering in one matmul; SVD-based
  matrix factorization reconstructs missing ratings from a compressed latent space — with the
  honest caveat that naive full-matrix SVD (treating missing as zero) is a real, known
  limitation of the simple approach shown here.
- Across all three domains, **zero new NumPy mechanics were introduced** — cosine similarity,
  broadcasting, axis reductions, `argpartition` top-K, and SVD from earlier chapters are simply
  redeployed against a new kind of data each time. Recognizing this is the actual meta-skill
  this trio of chapters is teaching.

---

## Mastery Exercises

**Exercise 1:** Why does `cosine_similarity_matrix` guard against zero-norm rows (`norms =
np.where(norms == 0, 1, norms)`) — what real input would trigger a division-by-zero otherwise,
in either the NLP or recommender-system context?

**Exercise 2 (Prediction):** For a `(10000,)` time series and `window=30`, predict the output
shape of `np.lib.stride_tricks.sliding_window_view(series, 30)`, and roughly how much *actual*
memory it uses compared to a naive materialized `(9971, 30)` array.

**Coding Challenge:** Implement `item_based_recommend(item_idx, ratings, top_k)` — the
item-based counterpart to this chapter's user-based `recommend_for_user`, using item-item
cosine similarity (computed on `ratings.T`) instead of user-user similarity. Compare its
recommendations for a specific user against the user-based approach's.

**Debugging Challenge:**
```python
series = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
windows = np.lib.stride_tricks.sliding_window_view(series, 3)
windows[0, 0] = 999.0   # a teammate is shocked this also changes series[0]!
print(series)
```
Explain exactly why this happens, referencing Chapter 8's view mechanism directly, and state
the one-line fix if independent mutation is actually needed.

**Interview Questions:**
1. Why does normalizing rows to unit length before a matrix multiplication compute cosine
   similarity for every pair simultaneously?
2. What's the algorithmic advantage of a `cumsum`-based rolling mean over a naive
   per-window-recomputation approach?
3. Explain, mechanically, why `sliding_window_view` can represent more "logical" elements than
   the array actually has in memory.
4. What's a known limitation of using naive full-matrix SVD directly for collaborative
   filtering, and what's the general fix?

**Quiz:**
- Q1: True/False — `np.diff(x)` returns an array the same length as `x`.
- Q2: In `sims @ ratings` for collaborative filtering, what do the shapes of `sims` and
  `ratings` need to be, and what shape does the result have?
- Q3: Which chapter's tool does `np.unique(all_words, return_inverse=True)` directly reuse for
  building a vocabulary?

---
 
